# SASRec Time-Aware BPI2012 Colab Train 09 (`Additive_sinusoidal`)

Stage 2 follow-up notebook for the new additive time-aware variant:
`Additive_sinusoidal`.

Goals:
- train `Additive_sinusoidal` with both `delta_prev_seconds` and `delta_start_seconds`
- use both backbones: `anchor_ml20`, `refine_ml50_do035`
- keep the usual 3 seeds: `42`, `2024`, `7`
- compare with completed Stage 2 runs, including `attention bias`
- display not only ranking metrics, but also task metrics such as `accuracy`, `macro_f1`, `top5_accuracy`, `top10_accuracy` when available


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
OUTPUT_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
ANCHOR_BUCKET_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10'
REFINE_BUCKET_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10'
REFINE_DSTART_BUCKET_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_refine_ml50_do035_b9_ndcg10'
REFINE_DSTART_CONT_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_deltastart_continuous_refine_ml50_do035_ndcg10'
ANCHOR_ATTN_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
REFINE_ATTN_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10'
SINUSOIDAL_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('SINUSOIDAL_NDCG10_OUTPUT_DIR:', SINUSOIDAL_NDCG10_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
SINUSOIDAL_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$SINUSOIDAL_NDCG10_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!rm -rf data/processed/bpi2012_complete_only
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

New runs to train:

- `anchor_ml20 + Additive_sinusoidal + delta_prev_seconds`
- `anchor_ml20 + Additive_sinusoidal + delta_start_seconds`
- `refine_ml50_do035 + Additive_sinusoidal + delta_prev_seconds`
- `refine_ml50_do035 + Additive_sinusoidal + delta_start_seconds`

Common setting:

- seeds: `42`, `2024`, `7`
- selection metric: `full_valid_ndcg@10`
- eval protocol: `both`


In [10]:
from pathlib import Path

comparison_run_groups = {
    'anchor_baseline': ['anchor_ml20_s42', 'anchor_ml20_s2024', 'anchor_ml20_s7'],
    'anchor_bucket_b9': ['timeaware_anchor_ml20_b9_s42', 'timeaware_anchor_ml20_b9_s2024', 'timeaware_anchor_ml20_b9_s7'],
    'anchor_attnbias_dstart_b9': ['attnbias_dstart_ml20_b9_s42', 'attnbias_dstart_ml20_b9_s2024', 'attnbias_dstart_ml20_b9_s7'],
    'anchor_sinusoidal_dprev': ['timeaware_dprev_sinusoidal_ml20_s42', 'timeaware_dprev_sinusoidal_ml20_s2024', 'timeaware_dprev_sinusoidal_ml20_s7'],
    'anchor_sinusoidal_dstart': ['timeaware_dstart_sinusoidal_ml20_s42', 'timeaware_dstart_sinusoidal_ml20_s2024', 'timeaware_dstart_sinusoidal_ml20_s7'],
    'refine_baseline': ['refine_ml50_do035_s42', 'refine_ml50_do035_s2024', 'refine_ml50_do035_s7'],
    'refine_bucket_b9': ['timeaware_refine_ml50_do035_b9_s42', 'timeaware_refine_ml50_do035_b9_s2024', 'timeaware_refine_ml50_do035_b9_s7'],
    'refine_dstart_bucket_b9': ['timeaware_dstart_refine_ml50_do035_b9_s42', 'timeaware_dstart_refine_ml50_do035_b9_s2024', 'timeaware_dstart_refine_ml50_do035_b9_s7'],
    'refine_dstart_continuous': ['timeaware_dstart_conti_refine_ml50_do035_s42', 'timeaware_dstart_conti_refine_ml50_do035_s2024', 'timeaware_dstart_conti_refine_ml50_do035_s7'],
    'refine_attnbias_dstart_b9': ['attnbias_dstart_ml50_do035_b9_s42', 'attnbias_dstart_ml50_do035_b9_s2024', 'attnbias_dstart_ml50_do035_b9_s7'],
    'refine_sinusoidal_dprev': ['timeaware_dprev_sinusoidal_ml50_do035_s42', 'timeaware_dprev_sinusoidal_ml50_do035_s2024', 'timeaware_dprev_sinusoidal_ml50_do035_s7'],
    'refine_sinusoidal_dstart': ['timeaware_dstart_sinusoidal_ml50_do035_s42', 'timeaware_dstart_sinusoidal_ml50_do035_s2024', 'timeaware_dstart_sinusoidal_ml50_do035_s7'],
}

scan_dirs = [
    Path(BASELINE_NDCG10_OUTPUT_DIR),
    Path(ANCHOR_BUCKET_NDCG10_OUTPUT_DIR),
    Path(REFINE_BUCKET_NDCG10_OUTPUT_DIR),
    Path(REFINE_DSTART_BUCKET_NDCG10_OUTPUT_DIR),
    Path(REFINE_DSTART_CONT_NDCG10_OUTPUT_DIR),
    Path(ANCHOR_ATTN_NDCG10_OUTPUT_DIR),
    Path(REFINE_ATTN_NDCG10_OUTPUT_DIR),
    Path(SINUSOIDAL_NDCG10_OUTPUT_DIR),
]

print('=' * 80)
print('Existing comparison runs')
for group_name, run_names in comparison_run_groups.items():
    print('-' * 80)
    print(group_name)
    for run_name in run_names:
        found = any((output_dir / run_name).exists() for output_dir in scan_dirs)
        print(run_name, 'EXISTS' if found else 'MISSING')


Existing comparison runs
--------------------------------------------------------------------------------
anchor_baseline
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS
--------------------------------------------------------------------------------
anchor_bucket_b9
timeaware_anchor_ml20_b9_s42 EXISTS
timeaware_anchor_ml20_b9_s2024 EXISTS
timeaware_anchor_ml20_b9_s7 EXISTS
--------------------------------------------------------------------------------
anchor_attnbias_dstart_b9
attnbias_dstart_ml20_b9_s42 EXISTS
attnbias_dstart_ml20_b9_s2024 EXISTS
attnbias_dstart_ml20_b9_s7 EXISTS
--------------------------------------------------------------------------------
anchor_sinusoidal_dprev
timeaware_dprev_sinusoidal_ml20_s42 EXISTS
timeaware_dprev_sinusoidal_ml20_s2024 EXISTS
timeaware_dprev_sinusoidal_ml20_s7 EXISTS
--------------------------------------------------------------------------------
anchor_sinusoidal_dstart
timeaware_dstart_sinusoidal_ml20_s42 EXISTS
tim

## Train `anchor_ml20 + Additive_sinusoidal + delta_prev_seconds`


### `timeaware_dprev_sinusoidal_ml20_s42`


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dprev_sinusoidal_ml20_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_prev_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dprev_sinusoidal_ml20_s42
epoch=1, loss=0.5364
epoch=2, loss=0.2320
epoch=3, loss=0.1671
epoch=4, loss=0.1393
epoch=5, loss=0.1207
valid [task], Top5Acc: 0.4868, Top10Acc: 0.7467, Acc: 0.1428, MacroF1: 0.1762
valid [full], NDCG@5: 0.6605, HR@5: 0.7867, NDCG@10: 0.7168, HR@10: 0.9707, MRR: 0.6440
valid [sampled], NDCG@5: 0.5520, HR@5: 0.5570, NDCG@10: 0.5617, HR@10: 0.5875, MRR: 0.5674
test [task], Top5Acc: 0.0925, Top10Acc: 0.6212, Acc: 0.0261, MacroF1: 0.0180
test [full], NDCG@5: 0.5746, HR@5: 0.8199, NDCG@10: 0.6343, HR@10: 0.9999, MRR: 0.51

### `timeaware_dprev_sinusoidal_ml20_s2024`


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dprev_sinusoidal_ml20_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_prev_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dprev_sinusoidal_ml20_s2024
epoch=1, loss=0.5362
epoch=2, loss=0.2364
epoch=3, loss=0.1722
epoch=4, loss=0.1381
epoch=5, loss=0.1199
valid [task], Top5Acc: 0.5001, Top10Acc: 0.7518, Acc: 0.1401, MacroF1: 0.1546
valid [full], NDCG@5: 0.6790, HR@5: 0.8077, NDCG@10: 0.7287, HR@10: 0.9631, MRR: 0.6603
valid [sampled], NDCG@5: 0.5556, HR@5: 0.5580, NDCG@10: 0.5638, HR@10: 0.5838, MRR: 0.5728
test [task], Top5Acc: 0.3513, Top10Acc: 0.7495, Acc: 0.0297, MacroF1: 0.0204
test [full], NDCG@5: 0.8138, HR@5: 0.9912, NDCG@10: 0.8169, HR@10: 1.0000, MRR: 0.

### `timeaware_dprev_sinusoidal_ml20_s7`


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_dprev_sinusoidal_ml20_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_prev_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dprev_sinusoidal_ml20_s7
epoch=1, loss=0.5064
epoch=2, loss=0.2315
epoch=3, loss=0.1701
epoch=4, loss=0.1361
epoch=5, loss=0.1178
valid [task], Top5Acc: 0.4869, Top10Acc: 0.6618, Acc: 0.0868, MacroF1: 0.1364
valid [full], NDCG@5: 0.6775, HR@5: 0.8294, NDCG@10: 0.7257, HR@10: 0.9780, MRR: 0.6510
valid [sampled], NDCG@5: 0.5540, HR@5: 0.5548, NDCG@10: 0.5599, HR@10: 0.5735, MRR: 0.5705
test [task], Top5Acc: 0.3438, Top10Acc: 0.7684, Acc: 0.0610, MacroF1: 0.0365
test [full], NDCG@5: 0.7837, HR@5: 0.9644, NDCG@10: 0.7936, HR@10: 0.9945, MRR: 0.729

## Train `anchor_ml20 + Additive_sinusoidal + delta_start_seconds`


### `timeaware_dstart_sinusoidal_ml20_s42`


In [11]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_sinusoidal_ml20_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dstart_sinusoidal_ml20_s42
epoch=1, loss=0.5417
epoch=2, loss=0.2377
epoch=3, loss=0.1699
epoch=4, loss=0.1383
epoch=5, loss=0.1200
valid [task], Top5Acc: 0.4897, Top10Acc: 0.6894, Acc: 0.1390, MacroF1: 0.1778
valid [full], NDCG@5: 0.6542, HR@5: 0.7720, NDCG@10: 0.7205, HR@10: 0.9790, MRR: 0.6452
valid [sampled], NDCG@5: 0.5550, HR@5: 0.5562, NDCG@10: 0.5608, HR@10: 0.5747, MRR: 0.5708
test [task], Top5Acc: 0.0767, Top10Acc: 0.6685, Acc: 0.0320, MacroF1: 0.0213
test [full], NDCG@5: 0.5611, HR@5: 0.8458, NDCG@10: 0.6110, HR@10: 0.9999, MRR: 0.4

### `timeaware_dstart_sinusoidal_ml20_s2024`


In [12]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_sinusoidal_ml20_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dstart_sinusoidal_ml20_s2024
epoch=1, loss=0.5341
epoch=2, loss=0.2353
epoch=3, loss=0.1724
epoch=4, loss=0.1382
epoch=5, loss=0.1198
valid [task], Top5Acc: 0.4919, Top10Acc: 0.6684, Acc: 0.1416, MacroF1: 0.1722
valid [full], NDCG@5: 0.6747, HR@5: 0.8291, NDCG@10: 0.7279, HR@10: 0.9881, MRR: 0.6496
valid [sampled], NDCG@5: 0.5460, HR@5: 0.5495, NDCG@10: 0.5542, HR@10: 0.5751, MRR: 0.5625
test [task], Top5Acc: 0.3301, Top10Acc: 0.7159, Acc: 0.0286, MacroF1: 0.0221
test [full], NDCG@5: 0.8179, HR@5: 0.9953, NDCG@10: 0.8196, HR@10: 1.0000, MRR: 0

### `timeaware_dstart_sinusoidal_ml20_s7`


In [13]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_sinusoidal_ml20_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dstart_sinusoidal_ml20_s7
epoch=1, loss=0.5013
epoch=2, loss=0.2301
epoch=3, loss=0.1704
epoch=4, loss=0.1365
epoch=5, loss=0.1181
valid [task], Top5Acc: 0.3909, Top10Acc: 0.5842, Acc: 0.1318, MacroF1: 0.1598
valid [full], NDCG@5: 0.6469, HR@5: 0.7665, NDCG@10: 0.7160, HR@10: 0.9756, MRR: 0.6403
valid [sampled], NDCG@5: 0.5552, HR@5: 0.5552, NDCG@10: 0.5576, HR@10: 0.5629, MRR: 0.5698
test [task], Top5Acc: 0.3368, Top10Acc: 0.6935, Acc: 0.0569, MacroF1: 0.0345
test [full], NDCG@5: 0.6839, HR@5: 0.8908, NDCG@10: 0.7138, HR@10: 0.9801, MRR: 0.62

## Train `refine_ml50_do035 + Additive_sinusoidal + delta_prev_seconds`


### `timeaware_dprev_sinusoidal_ml50_do035_s42`


In [14]:
!python src/train_sasrec.py \
  --run_name timeaware_dprev_sinusoidal_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_prev_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dprev_sinusoidal_ml50_do035_s42
epoch=1, loss=0.6141
epoch=2, loss=0.2764
epoch=3, loss=0.2091
epoch=4, loss=0.1736
epoch=5, loss=0.1503
valid [task], Top5Acc: 0.4868, Top10Acc: 0.6839, Acc: 0.0672, MacroF1: 0.0903
valid [full], NDCG@5: 0.6233, HR@5: 0.7164, NDCG@10: 0.6778, HR@10: 0.8884, MRR: 0.6252
valid [sampled], NDCG@5: 0.5520, HR@5: 0.5521, NDCG@10: 0.5532, HR@10: 0.5561, MRR: 0.5648
test [task], Top5Acc: 0.1788, Top10Acc: 0.4351, Acc: 0.0363, MacroF1: 0.0301
test [full], NDCG@5: 0.5168, HR@5: 0.6697, NDCG@10: 0.5660, HR@10: 0.8226, MRR

### `timeaware_dprev_sinusoidal_ml50_do035_s2024`


In [15]:
!python src/train_sasrec.py \
  --run_name timeaware_dprev_sinusoidal_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_prev_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dprev_sinusoidal_ml50_do035_s2024
epoch=1, loss=0.6150
epoch=2, loss=0.2794
epoch=3, loss=0.2047
epoch=4, loss=0.1658
epoch=5, loss=0.1445
valid [task], Top5Acc: 0.4892, Top10Acc: 0.7099, Acc: 0.0830, MacroF1: 0.1456
valid [full], NDCG@5: 0.7036, HR@5: 0.8896, NDCG@10: 0.7331, HR@10: 0.9827, MRR: 0.6580
valid [sampled], NDCG@5: 0.5585, HR@5: 0.5596, NDCG@10: 0.5625, HR@10: 0.5723, MRR: 0.5747
test [task], Top5Acc: 0.3383, Top10Acc: 0.8051, Acc: 0.0262, MacroF1: 0.0234
test [full], NDCG@5: 0.7766, HR@5: 0.9278, NDCG@10: 0.8012, HR@10: 1.0000, M

### `timeaware_dprev_sinusoidal_ml50_do035_s7`


In [16]:
!python src/train_sasrec.py \
  --run_name timeaware_dprev_sinusoidal_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_prev_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dprev_sinusoidal_ml50_do035_s7
epoch=1, loss=0.5913
epoch=2, loss=0.2778
epoch=3, loss=0.2061
epoch=4, loss=0.1739
epoch=5, loss=0.1520
valid [task], Top5Acc: 0.4827, Top10Acc: 0.7249, Acc: 0.0786, MacroF1: 0.1180
valid [full], NDCG@5: 0.6371, HR@5: 0.7505, NDCG@10: 0.7138, HR@10: 0.9782, MRR: 0.6365
valid [sampled], NDCG@5: 0.5512, HR@5: 0.5516, NDCG@10: 0.5529, HR@10: 0.5570, MRR: 0.5654
test [task], Top5Acc: 0.2035, Top10Acc: 0.7309, Acc: 0.0588, MacroF1: 0.0388
test [full], NDCG@5: 0.6391, HR@5: 0.8410, NDCG@10: 0.6861, HR@10: 0.9863, MRR:

## Train `refine_ml50_do035 + Additive_sinusoidal + delta_start_seconds`


### `timeaware_dstart_sinusoidal_ml50_do035_s42`


In [17]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_sinusoidal_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dstart_sinusoidal_ml50_do035_s42
epoch=1, loss=0.6154
epoch=2, loss=0.2774
epoch=3, loss=0.2108
epoch=4, loss=0.1742
epoch=5, loss=0.1498
valid [task], Top5Acc: 0.4962, Top10Acc: 0.7045, Acc: 0.0734, MacroF1: 0.0980
valid [full], NDCG@5: 0.6215, HR@5: 0.7058, NDCG@10: 0.6799, HR@10: 0.8882, MRR: 0.6280
valid [sampled], NDCG@5: 0.5575, HR@5: 0.5576, NDCG@10: 0.5579, HR@10: 0.5589, MRR: 0.5699
test [task], Top5Acc: 0.0935, Top10Acc: 0.4476, Acc: 0.0373, MacroF1: 0.0305
test [full], NDCG@5: 0.4981, HR@5: 0.7021, NDCG@10: 0.5290, HR@10: 0.8023, MR

### `timeaware_dstart_sinusoidal_ml50_do035_s2024`


In [18]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_sinusoidal_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dstart_sinusoidal_ml50_do035_s2024
epoch=1, loss=0.6154
epoch=2, loss=0.2754
epoch=3, loss=0.2018
epoch=4, loss=0.1644
epoch=5, loss=0.1434
valid [task], Top5Acc: 0.4804, Top10Acc: 0.6892, Acc: 0.0805, MacroF1: 0.1310
valid [full], NDCG@5: 0.6786, HR@5: 0.8288, NDCG@10: 0.7330, HR@10: 0.9928, MRR: 0.6547
valid [sampled], NDCG@5: 0.5575, HR@5: 0.5622, NDCG@10: 0.5655, HR@10: 0.5873, MRR: 0.5727
test [task], Top5Acc: 0.3819, Top10Acc: 0.7510, Acc: 0.0247, MacroF1: 0.0196
test [full], NDCG@5: 0.7566, HR@5: 0.8787, NDCG@10: 0.7970, HR@10: 1.0000, 

### `timeaware_dstart_sinusoidal_ml50_do035_s7`


In [19]:
!python src/train_sasrec.py \
  --run_name timeaware_dstart_sinusoidal_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --time_delta_column delta_start_seconds \
  --time_encoding sinusoidal \
  --time_sinusoidal_base 10000 \
  --output_dir "$SINUSOIDAL_NDCG10_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_sinusoidal_ndcg10/timeaware_dstart_sinusoidal_ml50_do035_s7
epoch=1, loss=0.5832
epoch=2, loss=0.2758
epoch=3, loss=0.2054
epoch=4, loss=0.1747
epoch=5, loss=0.1557
valid [task], Top5Acc: 0.4709, Top10Acc: 0.7359, Acc: 0.0782, MacroF1: 0.1197
valid [full], NDCG@5: 0.6339, HR@5: 0.7392, NDCG@10: 0.6854, HR@10: 0.9030, MRR: 0.6290
valid [sampled], NDCG@5: 0.5418, HR@5: 0.5435, NDCG@10: 0.5479, HR@10: 0.5627, MRR: 0.5569
test [task], Top5Acc: 0.0903, Top10Acc: 0.5773, Acc: 0.0449, MacroF1: 0.0326
test [full], NDCG@5: 0.5646, HR@5: 0.7624, NDCG@10: 0.6104, HR@10: 0.9025, MRR

## Rebuild result tables

This notebook reads result folders directly from `metrics_summary.json`.
Older Stage 2 runs may not contain all task metrics; in that case the corresponding cells appear as `NaN`.


In [20]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df_from_dirs(output_dirs):
    rows = []
    seen_runs = set()
    for output_dir in output_dirs:
        output_path = Path(output_dir)
        if not output_path.exists():
            continue
        for run_dir in output_path.iterdir():
            if not run_dir.is_dir() or run_dir.name in seen_runs:
                continue
            summary_path = run_dir / 'metrics_summary.json'
            config_path = run_dir / 'config.json'
            if not summary_path.exists() or not config_path.exists():
                continue
            summary = json.loads(summary_path.read_text(encoding='utf-8'))
            config = json.loads(config_path.read_text(encoding='utf-8'))
            row = {
                'run_name': summary.get('run_name'),
                'source_output_dir': str(output_path),
                'run_dir': str(run_dir),
                'completed_at': summary.get('completed_at'),
                'best_epoch': summary.get('best_epoch'),
                'checkpoint_best': summary.get('checkpoint_best'),
                'checkpoint_last': summary.get('checkpoint_last'),
                'metrics_history': summary.get('metrics_history'),
                'config_path': str(config_path),
                'metrics_summary': str(summary_path),
                'maxlen': config.get('maxlen'),
                'dropout_rate': config.get('dropout_rate'),
                'hidden_units': config.get('hidden_units'),
                'seed': config.get('seed'),
                'selection_metric': config.get('selection_metric'),
                'use_time_embedding': config.get('use_time_embedding', False),
                'use_time_attention_bias': config.get('use_time_attention_bias', False),
                'time_modeling_mode': config.get('time_modeling_mode'),
                'time_encoding': config.get('time_encoding'),
                'time_delta_column': config.get('time_delta_column'),
                'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
                'time_attention_bias_bucket_count': config.get('time_attention_bias_bucket_count'),
                'time_sinusoidal_base': config.get('time_sinusoidal_base'),
            }
            for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
                group = summary.get(group_name) or {}
                for mode, metrics in group.items():
                    for key, value in metrics.items():
                        row[f'{group_name}_{mode}_{key}'] = value
            rows.append(row)
            seen_runs.add(run_dir.name)
    return pd.DataFrame(rows)


In [21]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [22]:
scan_dirs = [
    BASELINE_NDCG10_OUTPUT_DIR,
    ANCHOR_BUCKET_NDCG10_OUTPUT_DIR,
    REFINE_BUCKET_NDCG10_OUTPUT_DIR,
    REFINE_DSTART_BUCKET_NDCG10_OUTPUT_DIR,
    REFINE_DSTART_CONT_NDCG10_OUTPUT_DIR,
    ANCHOR_ATTN_NDCG10_OUTPUT_DIR,
    REFINE_ATTN_NDCG10_OUTPUT_DIR,
    SINUSOIDAL_NDCG10_OUTPUT_DIR,
]

run_to_variant = {}
for run_name in ['anchor_ml20_s42', 'anchor_ml20_s2024', 'anchor_ml20_s7']:
    run_to_variant[run_name] = 'anchor_baseline'
for run_name in ['timeaware_anchor_ml20_b9_s42', 'timeaware_anchor_ml20_b9_s2024', 'timeaware_anchor_ml20_b9_s7']:
    run_to_variant[run_name] = 'anchor_bucket_b9'
for run_name in ['attnbias_dstart_ml20_b9_s42', 'attnbias_dstart_ml20_b9_s2024', 'attnbias_dstart_ml20_b9_s7']:
    run_to_variant[run_name] = 'anchor_attnbias_dstart_b9'
for run_name in ['timeaware_dprev_sinusoidal_ml20_s42', 'timeaware_dprev_sinusoidal_ml20_s2024', 'timeaware_dprev_sinusoidal_ml20_s7']:
    run_to_variant[run_name] = 'anchor_sinusoidal_dprev'
for run_name in ['timeaware_dstart_sinusoidal_ml20_s42', 'timeaware_dstart_sinusoidal_ml20_s2024', 'timeaware_dstart_sinusoidal_ml20_s7']:
    run_to_variant[run_name] = 'anchor_sinusoidal_dstart'

for run_name in ['refine_ml50_do035_s42', 'refine_ml50_do035_s2024', 'refine_ml50_do035_s7']:
    run_to_variant[run_name] = 'refine_baseline'
for run_name in ['timeaware_refine_ml50_do035_b9_s42', 'timeaware_refine_ml50_do035_b9_s2024', 'timeaware_refine_ml50_do035_b9_s7']:
    run_to_variant[run_name] = 'refine_bucket_b9'
for run_name in ['timeaware_dstart_refine_ml50_do035_b9_s42', 'timeaware_dstart_refine_ml50_do035_b9_s2024', 'timeaware_dstart_refine_ml50_do035_b9_s7']:
    run_to_variant[run_name] = 'refine_dstart_bucket_b9'
for run_name in ['timeaware_dstart_conti_refine_ml50_do035_s42', 'timeaware_dstart_conti_refine_ml50_do035_s2024', 'timeaware_dstart_conti_refine_ml50_do035_s7']:
    run_to_variant[run_name] = 'refine_dstart_continuous'
for run_name in ['attnbias_dstart_ml50_do035_b9_s42', 'attnbias_dstart_ml50_do035_b9_s2024', 'attnbias_dstart_ml50_do035_b9_s7']:
    run_to_variant[run_name] = 'refine_attnbias_dstart_b9'
for run_name in ['timeaware_dprev_sinusoidal_ml50_do035_s42', 'timeaware_dprev_sinusoidal_ml50_do035_s2024', 'timeaware_dprev_sinusoidal_ml50_do035_s7']:
    run_to_variant[run_name] = 'refine_sinusoidal_dprev'
for run_name in ['timeaware_dstart_sinusoidal_ml50_do035_s42', 'timeaware_dstart_sinusoidal_ml50_do035_s2024', 'timeaware_dstart_sinusoidal_ml50_do035_s7']:
    run_to_variant[run_name] = 'refine_sinusoidal_dstart'

df_all = rebuild_df_from_dirs(scan_dirs)
df_compare = df_all[df_all['run_name'].isin(run_to_variant)].copy()
df_compare['variant'] = df_compare['run_name'].map(run_to_variant)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

display_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric',
    'time_modeling_mode', 'time_encoding', 'time_delta_column', 'time_attention_bias_bucket_count', 'time_sinusoidal_base',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_ndcg@5', 'best_valid_full_hr@5', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5', 'best_test_at_best_valid_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10', 'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5', 'best_valid_sampled_mrr',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10', 'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5', 'best_test_at_best_valid_sampled_mrr',
    'best_valid_task_accuracy', 'best_valid_task_macro_f1', 'best_valid_task_top5_accuracy', 'best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_accuracy', 'best_test_at_best_valid_task_macro_f1', 'best_test_at_best_valid_task_top5_accuracy', 'best_test_at_best_valid_task_top10_accuracy',
]
existing_display_cols = [c for c in display_cols if c in df_compare.columns]
df_compare[existing_display_cols]


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,time_modeling_mode,time_encoding,time_delta_column,time_attention_bias_bucket_count,time_sinusoidal_base,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_mrr,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_top5_accuracy,best_valid_task_top10_accuracy,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_top5_accuracy,best_test_at_best_valid_task_top10_accuracy
0,attnbias_dstart_ml20_b9_s7,7,anchor_attnbias_dstart_b9,20,0.20,full_valid_ndcg@10,attention_bias,raw,delta_start_seconds,7.0,NaN,0.731681,0.933234,0.694816,0.812322,0.674079,0.833933,1.000000,0.833460,0.998642,0.776636,0.584594,0.673558,0.550907,0.567329,0.571303,0.408188,0.553189,0.349894,0.366486,0.395330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,attnbias_dstart_ml20_b9_s42,42,anchor_attnbias_dstart_b9,20,0.20,full_valid_ndcg@10,attention_bias,raw,delta_start_seconds,7.0,NaN,0.731547,0.976377,0.692852,0.854887,0.657784,0.857427,1.000000,0.844950,0.958941,0.810092,0.553607,0.600217,0.535631,0.543183,0.557929,0.535711,0.621374,0.504964,0.524491,0.530831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,attnbias_dstart_ml20_b9_s2024,2024,anchor_attnbias_dstart_b9,20,0.20,full_valid_ndcg@10,attention_bias,raw,delta_start_seconds,7.0,NaN,0.727796,0.977380,0.682637,0.839902,0.653203,0.815191,1.000000,0.800095,0.953875,0.753665,0.551441,0.581497,0.539500,0.543537,0.560101,0.369433,0.517223,0.311595,0.332384,0.355765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,anchor_ml20_s7,7,anchor_baseline,20,0.20,full_valid_ndcg@10,NaN,NaN,delta_prev_seconds,NaN,NaN,0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,anchor_ml20_s42,42,anchor_baseline,20,0.20,full_valid_ndcg@10,NaN,NaN,delta_prev_seconds,NaN,NaN,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,anchor_ml20_s2024,2024,anchor_baseline,20,0.20,full_valid_ndcg@10,NaN,NaN,delta_prev_seconds,NaN,NaN,0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,timeaware_anchor_ml20_b9_s7,7,anchor_bucket_b9,20,0.20,full_valid_ndcg@10,NaN,NaN,delta_prev_seconds,NaN,NaN,0.743844,0.960530,0.714598,0.867397,0.678884,0.843990,1.000000,0.834025,0.971564,0.790520,0.582482,0.666984,0.550773,0.566848,0.573250,0.386793,0.556218,0.322537,0.352208,0.366348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,timeaware_anchor_ml20_b9_s42,42,anchor_bucket_b9,20,0.20,full_valid_ndcg@10,NaN,NaN,delta_prev_seconds,NaN,NaN,0.715128,0.981350,0.642260,0.750340,0.638622,0.813155,1.000000,0.778245,0.893199,0.753896,0.565702,0.581509,0.559273,0.560707,0.575554,0.269814,0.520394,0.187604,0.266343,0.220679,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,timeaware_anchor_ml20_b9_s2024,2024,anchor_bucket_b9,20,0.20,full_valid_ndcg@10,NaN,NaN,delta_prev_seconds,NaN,NaN,0.724989,0.975136,0.655030,0.761685,0.651891,0.810350,1.000000,0.777245,0.897405,0.749042,0.572362,0.618261,0.556768,0.569764,0.571922,0.426353,0.538939,0.380476,0.391281,0.420197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,timeaware_

In [23]:
summary_metric_cols = [
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_ndcg@5', 'best_valid_full_hr@5', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5', 'best_test_at_best_valid_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10', 'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5', 'best_valid_sampled_mrr',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10', 'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5', 'best_test_at_best_valid_sampled_mrr',
    'best_valid_task_accuracy', 'best_valid_task_macro_f1', 'best_valid_task_top5_accuracy', 'best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_accuracy', 'best_test_at_best_valid_task_macro_f1', 'best_test_at_best_valid_task_top5_accuracy', 'best_test_at_best_valid_task_top10_accuracy',
]
summary_metric_cols = [c for c in summary_metric_cols if c in df_compare.columns]
summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_mrr           best_valid_task_accuracy           best_valid_task_macro_f1           best_valid_task_top5_accuracy           best_valid_task_top10_accuracy           best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_top5_accuracy           best_test_at_best_valid_task_top10_accuracy          
                                             mean       std                  mean       std                   mean       std                 mean       std                mean       std                                 mean       std                               mean       std                                mean       std                              mean       std                             mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                                    mean       std                                  mean       std                                   mean       std                                 mean       std                                mean       std                     mean       std                     mean       std                          mean       std                           mean       std                                  mean       std                                  mean       std                                       mean       std                                        mean       std
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
anchor_attnbias_dstart_b9                0.730341  0.002206              0.962330  0.025203               0.690102  0.006539             0.835704  0.021591            0.661689  0.010972                             0.835517  0.021162                           1.000000  0.000000                            0.826168  0.023299                          0.970486  0.024515                         0.780131  0.028375                   0.563214  0.018547                 0.618424  0.048656    

Interpretation guide:

- main comparison metric: `best_test_at_best_valid_full_ndcg@10`
- compare `Additive_sinusoidal` against completed baseline / additive / attention-bias runs
- task metrics (`accuracy`, `macro_f1`, `top5_accuracy`, `top10_accuracy`) are shown when the corresponding run folders already contain them
